# Combine Dense and Sparse Retriever


In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever ## Sparse Retriever
from langchain_classic.retrievers import EnsembleRetriever

C:\Users\ARKAN\AppData\Local\Temp\ipykernel_25268\3643034931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
e:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Dense Retriever (FAISS + Embedding model)
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs,embedding=embedding_model)
dense_retriever = dense_vectorstore.as_retriever() # Dense Retriever


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2758.58it/s]


In [3]:
# Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3

## Combine with EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7,0.3]
)

In [4]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000239065DF190>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000023906A89490>, k=3)], weights=[0.7, 0.3])

In [4]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


In [5]:
# RAG Pipelines with hybrid Search
from langchain.chat_models import init_chat_model
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
## initalize chat model
llm = init_chat_model(model="openai/gpt-oss-120b",temperature=0.7, model_provider="groq")

In [8]:
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

In [9]:
rag_chain = create_retrieval_chain(hybrid_retriever, document_chain)

In [10]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 Below is a high‑level roadmap for building an application that leverages large language models (LLMs) – using the pieces mentioned in your context (LangChain, agents, retrievers, and Pinecone).

---

## 1. Define the product goal
- **What does the app do?**  
  e.g., answer user questions, generate code, summarize documents, act as a personal assistant, etc.  
- **What data does it need?**  
  Structured knowledge (databases, APIs), unstructured text (documents, webpages), or pure generation.

---

## 2. Choose the LLM provider
Pick a model that fits your latency, cost, and capability needs (e.g., OpenAI GPT‑4, Anthropic Claude, Llama‑2, etc.). Most LangChain integrations let you swap the model with a single line of code.

```python
from langchain.llms import OpenAI   # or AzureOpenAI, Anthropic, etc.
llm = OpenAI(model="gpt-4", temperature=0.2)
```

---

## 3. Set up a **LangChain** pipeline
LangChain gives you reusable building blocks:

| Block | Typical role | Example cod

# Re-ranking Hybrid Search Strategies

In [24]:
from langchain_community.document_loaders import TextLoader
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

In [16]:
# Load the Documents
text_loader = TextLoader("E:\RAG\Data\langchain_sample.txt")
raw_docs=text_loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, 

In [17]:
# FAISS and Huggingface embeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model="all-MiniLM-L6-V2")
vector_store = FAISS.from_documents(documents=docs, embedding=embedding)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4809.98it/s]


In [18]:
retriever = vector_store.as_retriever()

In [19]:
## Initialize the LLM 
llm = init_chat_model(model="openai/gpt-oss-120b",temperature=0.7, model_provider="groq")

In [40]:
llm.invoke("hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". Simple greeting. Should respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 72, 'total_tokens': 104, 'completion_time': 0.065936663, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.002967315, 'prompt_tokens_details': None, 'queue_time': 0.281280283, 'total_time': 0.068903978}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2bdf-b4fb-75a3-9479-1ec3070c0cf4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 32, 'total_tokens': 104, 'output_token_details': {'reasoning': 14}})

In [33]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [34]:
query="How can i use langchain to build an application with memory and tools?"

In [35]:
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='40c9affe-6087-4b23-b2d4-3cfc0e7b3474', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='2c4fbdc6-d3bf-4967-8fa5-56ee20434d6c', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='9fcc52b8-1f8e-4509-a3cb-6a5363982cb7', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain integrates with many t

In [44]:
chain = prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs':

In [45]:
retrieved_docs

[Document(id='40c9affe-6087-4b23-b2d4-3cfc0e7b3474', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='2c4fbdc6-d3bf-4967-8fa5-56ee20434d6c', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='9fcc52b8-1f8e-4509-a3cb-6a5363982cb7', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain integrates with many t

In [46]:
doc_lines = [f"{i+1}. {doc.page_content}"for i, doc in enumerate(retrieved_docs)]
formated_docs = "\n".join(doc_lines)
formated_docs
type(formated_docs)

str

In [47]:
response = chain.invoke({"question": query, "documents": formated_docs})
response

'1,2,4,3'

In [51]:
# parse the response and re-rank the retrieved documents
indices = [int(s.strip())-1 for s in response.split(",") if s.strip().isdigit()]
indices

[0, 1, 3, 2]

In [53]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='40c9affe-6087-4b23-b2d4-3cfc0e7b3474', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='2c4fbdc6-d3bf-4967-8fa5-56ee20434d6c', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='abf9a848-3709-4b42-9a78-44110191e1b7', metadata={'source': 'E:\\RAG\\Data\\langchain_sample.txt'}, page_content='FAISS is a popular library used 

In [54]:
# Step 6: Show results
print("\n📊 Final Reranked Results:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}:\n{doc.page_content}")


📊 Final Reranked Results:

Rank 1:
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank 2:
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

Rank 3:
FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes, which makes it scalable for large document stores.
Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering w

# Maximal Marginal Relevance (MMR)

In [8]:
from langchain_community.document_loaders import TextLoader
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:
## load the text
loader = TextLoader("E:\RAG\Data\langchain_rag_dataset.txt")
docs = loader.load()
docs

[Document(metadata={'source': 'E:\\RAG\\Data\\langchain_rag_dataset.txt'}, page_content="LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.\nBM25 and vector-based retrieval can be combined in LangChain to support hybrid retrieval strategies.\nFAISS is a high-performance library for similarity search 

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'E:\\RAG\\Data\\langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'E:\\RAG\\Data\\langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'E:\\RAG\\Data\\langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instru

In [12]:
## initialize embedding and vector store
embedding = HuggingFaceEmbeddings(model="all-MiniLM-L6-V2")

vectorstore = FAISS.from_documents(chunks, embedding=embedding)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3264.17it/s]


In [16]:
### Create MMR Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwarg={"k":3}
)

In [17]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")
llm = init_chat_model(model="openai/gpt-oss-120b",temperature=0.7, model_provider="groq")

In [19]:
# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [20]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 **LangChain’s support for agents**

* **Tool‑driven agents** – LangChain lets an LLM act as an *agent* that can decide, at each step, which tool to invoke (e.g., a calculator, a web‑search API, a custom function, a database query, etc.).  
* **Dynamic tool selection** – The agent receives a prompt describing the available tools and the current task, then the LLM outputs a “tool‑call” name and arguments. LangChain routes that request to the appropriate function, feeds the result back to the model, and repeats until the task is finished.  
* **External integration** – Because the tools can be any callable code, agents can interact with external APIs, databases, or other services, extending the raw LLM’s capabilities far beyond text generation.  
* **Built‑in agent classes** – LangChain provides ready‑made agent implementations (e.g., `ZeroShotAgent`, `ReactAgent`, `OpenAIFunctionsAgent`) that encapsulate the “think‑act‑observe” loop, making it easy to plug in new tools without

In [21]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='9264f7e4-f05d-4d7c-a60c-829aa70c057b', metadata={'source': 'E:\\RAG\\Data\\langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='cf3e8f1c-fa16-44b6-87b3-1868a0154d09', metadata={'source': 'E:\\RAG\\Data\\langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='b04d4763-04ae-4e00-a7a1-9b0fda1bb1c9', metadata={'source': 'E:\\RAG\\Data\\langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that dec